In [ ]:
#新闻推送，查看负面舆情等

import jieba.posseg as pseg
import jieba
import os
from pyltp import NamedEntityRecognizer, Postagger, Segmentor
from pyltp import SentenceSplitter
import re

In [ ]:
# 加载LTP模型
LTP_DATA_DIR = '\ltp_data_v3.4.0'  # LTP模型目录的路径
cws_model_path = os.path.join(LTP_DATA_DIR, 'cws.model')  # 分词模型路径，模型名称为`cws.model`
pos_model_path = os.path.join(LTP_DATA_DIR, 'pos.model')  # 词性标注模型路径，模型名称为`pos.model`
ner_model_path = os.path.join(LTP_DATA_DIR, 'ner.model')  # 命名实体识别模型路径，模型名称为`ner.model`
segmentor = Segmentor(cws_model_path)  # 加载初始化分词模型
postagger = Postagger(pos_model_path)  # 加载初始化词性标注模型
recognizer = NamedEntityRecognizer(ner_model_path)  # 加载初始化命名实体识别模型

In [ ]:
def get_entity(text):
    """
    对新闻文本进行实体识别，返回人名、地名、机构名等实体
    """
    # 使用jieba进行分词
    jieba.enable_paddle()
    words = pseg.cut(text,jieba.enable_paddle())
    

    # 将分词结果整理成LTP需要的输入格式
    word_list = []
    pos_list = []
    for word, pos in words:
        word_list.append(word)
        pos_list.append(pos)

    # 对文本进行分句
    sentences = SentenceSplitter.split(text.strip())
    entity_list = []
    for sentence in sentences:
        sentence = sentence.strip()
        if sentence:
            # 分词
            words = list(segmentor.segment(sentence))
            # 词性标注
            postags = list(postagger.postag(words))
            # 命名实体识别
            netags = list(recognizer.recognize(words, postags))
            # 整理结果
            for word, tag in zip(words, netags):
                if tag != 'O':
                    entity_list.append(word)

    # 关闭LTP模型
    segmentor.release()
    postagger.release()
    recognizer.release()

    return words,postags,netags,entity_list

In [ ]:
import paddle

# Enable static graph mode
paddle.enable_static()

# 测试
text = '平安普惠山西分公司：金融反诈进社区，守护您的“钱袋子”'
words,postags,netags,entity_list = get_entity(text)
print(entity_list)

In [ ]:
import jieba
import jieba.posseg as pseg
import re
from textrank4zh import TextRank4Sentence

# 加载自定义词典，提供人名和职务
# jieba.load_userdict('custom_dict.txt')

# 定义人名和职务标签
# person_tag = ['nr', 'nrfg']
# title_tag = [ 'nt','nr','v','n']
person_tag = ['PER']
title_tag = [ 'ORG']
job_tag = ['n','ORG','w','nz','v']

# 定义待提取文本
text = '中国人民银行支付结算司司长谢众14日在中信“党费通”产品发布暨客户签约仪式上指出，2018年中国人民银行将继续大力推动我国非现金支付环境的建设，指导市场主体对企业、居民提供更加安全、便捷、高效的支付产品。同时，持续做好正本清源工作，进一步加大市场监管力度，防范化解金融风险，对各类违规支付行为的整顿处罚绝不手软，维护支付市场的公平竞争秩序，使移动支付服务更好地发挥对实体经济和普惠金融发展的促进作用。谢众表示，中信“党费通”率先使用移动支付技术实现党费交纳和管理，为各级党组织和广大党员提供创新服务，实现了为新时期党建工作赋能。中国人民银行一直致力于推动我国移动支付服务的创新发展，指导主导市场主体持续推动非现金支付工具的应用。去年以来，中国银联和商业银行正在积极组织开展移动支付便民示范工程，统筹推进移动支付便民利用和受理环境改造等工作，部署打造银行业统一移动支付APP，广泛覆盖各类移动终端产品和移动支付技术，深度结合地铁、公交、校企园区和公用事业缴费等人民群众衣食住行各类支付产品，推动普惠金融发展。中信银行党委书记、董事长李庆萍介绍，作为集党费缴纳、管理、教育、服务于一体的综合管理平台，“党费通”在实现多场景运用和多方式支付的同时，兼具安全性和仪式感，是中信集团和中信银行打造智慧党建的重要成果。在14日的签约环节，中信银行与10家“党费通”产品使用单位和党建工作平台建设单位分别签约。'
# text = '据银保监会官网消息，10日，中国人民银行党委书记、银保监会主席郭树清会见了彼得森国际经济研究所(PIIE)所长亚当·珀森(Adam Posen)先生一行，双方就货币信贷、金融科技和金融业对外开放等问题进行了交流。 郭树清表示，当前中国货币信贷供应与经济增长需求基本保持一致，但也还需进一步畅通传导机制，稳步推进结构性去杠杆，提高资金使用效率。　郭树清指出，随着大数据、云计算、区块链和人工智能等新技术广泛应用，金融科技近年来在中国发展非常迅速，在服务小微企业、普惠金融，提高服务效率等方面发挥了重要作用。但同时我们也密切关注其对金融行业的影响，尤其是营运模式变化、消费者保护，以及网络系统安全等。 郭树清同时指出，金融科技必须遵循统一的监管规则和风险防控标准。关于金融业对外开放，郭树清进一步表示，中国金融业对外开放的大门会越开越大，欢迎各类外国金融机构来华与中资金融机构在同一平台、在各个领域开展合作，共同竞争，实现“多赢”。'
# text = '2018年以来，人民银行拉萨中心支行组织西藏金融系统认真学习习近平总书记在民营企业座谈会上的重要讲话精神，贯彻落实党中央、国务院关于缓解民营小微企业融资难融资贵的决策部署，按照区党委政府、人总行关于民营小微企业工作要求，强化政策引导，健全组织体系，深化“放管服”改革，推动金融科技创新，努力纾解民营小微企业融资难问题，持续加大对民营小微企业的金融支持力度。</P><P class=Para>　　坚持问题导向，推出民营小微金融服务专项政策。组织全区金融系统认真领会习近平总书记在民营企业座谈会上的指示精神，要求西藏金融机构切实提高政治站位，转变思想观念，把支持民营企业发展作为支持实体经济、促进自身健康发展的重中之重。坚持问题导向，深入一线，掌握小微企业一手资料，根据调研情况和辖区实际制定印发了一系列小微企业金融服务专项政策，疏通政策传导“阻碍”。2018年以来，仅拉萨中支党委围绕金融支持小微民营企业发展，深入企业、农牧区、基层累计调研达20余次，足迹踏遍了西藏74个县。坚持服务提升，健全民营小微金融服务保障体系。统筹民营小微企业金融服务工作，组织全区金融系统健全普惠金融组织体系，完善支持民营小微企业专营机构和专门工作团队设置。截至目前，西藏9家银行机构设立了普惠金融事业部，6家银行机构设立了小微专营团队和部门。坚持精准滴灌，引导金融机构增加民营小微信贷投放。鉴于辖区法人金融机构规模偏小、实力较弱，在总行的大力支持下，破格将西藏银行、堆龙民泰村镇银行纳入扶贫再贷款发放对象，加大对地方法人金融机构民营小微企业信贷投放的流动性支持。截至目前，累计发放再贷款资金12.93亿元，运用再贷款资金发放212笔贷款。</P><P class=Para>　　坚持作风改进，提供民营小微企业个性化服务。拉萨中支组织银行业金融机构实施“百家银行帮百企、百名行长进百企”金融结对帮扶计划，组织我区15家银行机构近100位领导干部结对帮扶余家民营和小微企业，深入了解经营状况、融资需求，为企业定制差异化的融资方案，为290余家企业直接提供信贷资金19.41多亿元，解决融资需求。</P><P class=Para>　　坚持便利企业，搭建银企融资对接沟通桥梁。2018年以来，拉萨中支牵头组织大型银企对接会3次，累计授信、放贷金额达290.25亿元，涉及300多家民营小微企业。试点运行西藏首个银企融资信息对接服务平台，纾解银企信息不对称的瓶颈问题，民营小微企业贷款办理时间平均缩短15天左右。加大应收账款融资服务平台推广力度，提升征信服务民营企业小微企业的能力。2019年上半年平台累计促成中小微企业融资业务11笔、融资金额达24.02亿元，同比分别增长266.67%、344.81%。坚持金融创新，提升民营小微信贷可获得性。督导辖区金融系统推广“微捷贷”、“小微快贷”等线上小微信贷产品。2018年以来，小微企业创新信贷产品累计发放1636笔、35.77亿元。鼓励金融机构结合地方实际创新金融产品，如工行西藏分行创设“藏宿贷”，专门用于支持林芝农牧民家庭旅馆升级改扩建、装修或采购设施，2018年以来，林芝地区18家家庭客栈通过“藏宿贷”支持完成升级改造并投入使用，2019年桃花节期间实现收入452万元，较2018同期增收300%。坚持环境优化，加强政策宣传普及和营商环境整改。牵头开展大剂量、全方位、多角度、立体式的民营小微企业金融政策及服务宣传。截至目前，省地两级新闻媒体累计开展藏汉双语宣传及访谈近二百余次。推动金融机构实施优化营商环境行动，引导金融系统精简办贷资料、优化信贷流程、明确办理时限、推行阳光办贷。</P><P class=Para>　　当前，民营小微企业金融服务呈“量增、价稳、面扩”的特点。6月末，小微企业贷款余额1049.17亿元，同比增长26.2%，高于同期各项贷款增速20.37个百分点。其中，普惠口径小微企业贷款余额64.93亿元，同比增长2.4倍。融资成本趋于稳定。6月末，金融机构新发放的1000万元以下小微企业贷款加权平均利率约为2.63%，与年初基本持平。6月末，小微企业贷款户数2769户，较上年末多增600户。金融机构对普惠口径小微主体授信7198户，比上年末增加1840户，增长34.3%。金融有力支持了西藏民营小微企业的发展。'
# text = '与民营企业共成长，与民营经济同发展。　招商银行作为国内第一家由企业法人出资创立的商业银行，与民营企业之间更为亲近。自2000年3月成立以来，招商银行福州分行一直把民营企业和中小企业作为重点服务对象，秉持“因您而变”的服务理念，倾力支持福州民营经济发展壮大。　近年来，招商银行福州分行积极响应国家关于扶持民营经济发展战略，从创新服务模式、优化服务路径、拓展服务内容等方面频出实招，支持民营企业发展。截至2019年8月末，招商银行福州分行民营企业贷款余额111.33亿元，余额占总对公贷款金额的47.15%，民营企业贷款当年累放额260.29亿元，占2019年累计投放额的63.68%。　　新机制激发新活力　福建某科技企业是一家新三板挂牌高新技术企业，集IT综合研发、网络技术及软件产品开发、智慧教育类产品研发为一体，具有重技术研发投入、轻资产投入等特点。近两年，由于业务发展迅速，企业需要更多的资金。然而，由于缺乏抵质押物，企业的融资之路并不顺畅。　知晓这一情况后，招行福州分行迅速成立专项服务小组。客户经理主动上门了解企业情况和需求，沟通分析认为企业完全符合招行“高新贷”产品特性要求，便快速为企业办理融资、发放贷款。“招行福州分行为我们设计了全方位专属融资综合方案，提交融资申请后，贷款很快就发放了。”该科技企业负责人说道。　　除轻资产、无抵质押物外，中小企业“小、短、急”的融资需求特点也是造成其融资困难的一大原因。传统的信贷流程和条件无法满足企业的融资需求。为此，去年以来，招行结合市场需求，进一步优化信贷审批流程，精简业务审批耗时环节，缩短信贷审批时间，部分产品实现全流程审批平均用时3个工作日，有效提升了金融服务质效。　　在传统信贷审批手段基础上，招行福州分行还借助总行的金融科技力量，对外接入第三方大数据，如海关数据、纳税数据、银行流水、上下游交易数据等，开发便捷、快速的信贷系统，为企业提供更便捷的融资及其他金融服务。　金融助力，企业发展更有底气。长期以来，招行致力于服务科技创新企业，与福建省财政厅、福建省科技厅等政府部门持续保持良好的合作关系，先后推出“高新贷”“政采贷”“科技贷”等一系列小微企业专属融资产品；更是在业界创新推出“千鹰展翼创新型成长企业培育计划”，与律所、券商等合作机构搭建千鹰展翼“朋友圈”合作联盟，为高新技术企业提供多元化金融服务，全面助力中小企业快速发展。　　新科技,带来新速度　互联网时代，金融业务数字化大大提升了银行机构服务民营企业的速度和效率。　基于在互联网时代的先发优势，2018年，招行推出了“企业APP”（企业端的手机银行），面向各类型企业用户提供开放式移动服务。该APP综合运用人脸识别、手机证书、AI人工智能等FinTech金融科技成果，实现企业级的移动端刷脸支付，单笔支付金额最高可达500万元；同时实现了各类移动场景下的安全、智能、便捷的免Ukey交易，可为企业客户提供便捷业务办理服务。　“只要下载注册手机APP，企业主就可以随时、随地、随手办理金融交易、财务支付等业务，同时还能享受福州专区特色云服务，参与到一系列深入企业经营的金融及非金融场景化服务。”招行福州分行有关负责人介绍，截至2019年8月末，“企业APP”的企业用户已超过9420户。　数据多跑路，企业少跑腿。近年来，招行福州分行借助金融科技力量，着力提升金融服务民营企业、小微企业的效率。为满足小微客户融资“短、小、频、急”的要求，招行将金融科技应用于小微企业业务的申请、审批及贷后服务的每个环节，充分体现了招行“金融科技银行”的战略定位。　招行福州分行有关负责人介绍，2015年，该行率先推出PAD移动作业平台，可现场直接进行预审批，1分钟获取预审批结果，24小时内完成审批，大幅提升了企业客户申请贷款的时效；建立“一个中心批全国”信贷工厂集中审批模式，持续优化贷款审批前的中台作业流转程序，为小微金融服务搭建全程“高速公路”；依托手机银行打造线上“轻服务”，方便小微企业客户自助快捷办理，同时上线微信预约办理功能，减少客户现场等候时间，改善客户体验。截至2019年8月，招行福州分行已累计服务小微客户12019（含个人11058)户，近3年累计发放小微贷款350.09亿元（含个人221.92亿元）。　　新服务,提供新体验　　为更好地服务民营经济发展，践行普惠金融，招行福州分行今年推出多项举措优化企业开户，助力营商环境改善，获得了众多民营企业频频“点赞”。　取消企业银行账户许可，对于民营企业和小微企业的发展意义重大。今年4月28日，为落地“取消企业银行账户许可”相关要求，招行福州分行以Fintech引领金融服务创新，全面提升银行账户服务效率，持续优化企业开户服务。该行有关负责人表示，取消企业银行账户核准，不仅开户许可证不再作为企业办理其他事务的证明文件或依据，更进一步压缩了企业账户开办时间。　　因需而生，因势而变。自成立以来，招行福州分行始终秉持“以客户为中心”的服务理念，根据企业客户的实际需求与痛点、难点，设计专属的融资方案。对于招行“因您而变”的小微金融服务，福州小微企业主熊女士十分满意，“我在外地出差，就能把在福州办的贷款成功转贷，这多亏了招行客户经理跨城上门帮忙。”　据悉，熊女士在招行的贷款即将到期，但当时企业的经营资金尚未回笼，她短时间内无法调用大笔资金用于归还贷款。由于在外地出差，她也无法在指定期限内回到福州办理转贷业务。招行客户经理小林了解情况后，乘坐高铁到南平市为熊女士上门签署转贷材料，最终在时效内帮助熊女士转贷成功，保障了熊女士正常的生意运作和良好的征信记录。　　为进一步提高客户服务体验，招行福州分行还积极响应福建省地方金融监督管理局、福建省工业和信息化厅号召，开展了“百名行长进企业”活动，深入调研走访企业，通过与企业主面对面沟通交流，掌握企业及行业的发展现状，了解企业生产经营情况、融资需求；同时，大力宣传该行普惠金融政策及重点金融产品，深入了解企业客户对招行产品及服务的意见和建议，切实解决企业“融资难、融资贵”问题，将服务民营企业落到实处，助力民营企业健康发展。　下一阶段，招行福州分行将继续高举发展的旗帜，不断提高自身能力，加大金融科技力量的投入，不断探索新产品与新服务，以满足民营企业的金融服务需求，全面支持民营企业发展。'
#text = '北京银行在改革开放的大潮中孕育而生，经过22年的发展，通过“更名、引资、改制、上市”，从一家基础薄弱的小银行成长为全球百强银行，成为中国金融改革开放的见证和缩影。今年上半年，该行坚持以高质量发展为主线，以建设“十大银行”为战略目标，经营业绩、业务规模、管理效能实现全方位提升，谱写了改革发展的崭新篇章。8月30日，该行发布2018年半年报，向广大投资者交出一靓丽答卷。据悉，北京银行始终坚持一手抓企业党建，一手抓业务发展，狠抓全面从严治党，扎实推进党的领导和公司治理深度融合，充分发挥各级党组织的政治核心和战斗堡垒作用，将“党建+”融入改革发展的全过程和业务发展各领域；深入开展行内巡察和巡察“回头看”工作，有效运用监督执纪“四种形态”，维护风清气正的发展环境。在党的坚强领导和党建创新引领下，北京银行将保持政治定力和激发市场活力有机统一，在服务实体经济发展、服务首都城市建设的探索实践中，打造了首都金融业的一张亮丽名片。　经营业绩稳步增长中小银行“领头羊”地位凸显　截至2018年6月末，北京银行表内外总资产达到3.22万亿元，成立22年增长160倍；净资产1839亿元，成立22年增长180倍；资产总额2.49万亿元，较年初增长6.7%；半年实现净利润119亿元，同比增长6.60%；贷款总额增长12.8%，存款增长9.4%；净息差1.82%，较上季环比提升10BP；成本收入比21.73%，半年实现人均创利82万元，各项经营指标继续保持同业优秀水平。　基于对高质量发展的探索，该行品牌价值持续提升，连续五年跻身世界百强银行之列，位居首都金融业第一位。在2018年7月英国《银行家》杂志公布的全球前1000强银行排名中，北京银行按照一级资本排名第63位，较去年大幅提升10位，位居首都金融业第一位，连续五年跻身世界百强银行之列；在世界品牌实验室品牌价值排行榜中，北京银行品牌价值提升至449亿元，排名中国银行业第7位。先后获得了《金融时报》“年度最佳中小企业金融服务银行”，中国银行业协会“中国银行业文明规范服务工作突出贡献奖”，《中华工商时报》“年度品牌影响力银行”《新京报》“年度市民信赖品质银行”，连续10次荣获《亚洲银行家》“中国最佳城市商业零售银行”大奖。　转型取得新突破各业务线亮点纷呈　从各项业务经营数据来看，北京银行各业务条线均呈现良好的发展态势。　公司业务方面，该行加大稳存增存力度，公司存款迈上1万亿元台阶；中标人社部“央保卡”发行服务合作银行，在社保领域打开与中央部委合作渠道；发布“e商融”交易市场综合服务方案，打造“供应链金融+资金监管”线上创新业务模式；探索小微企业网络融资新模式，推出“永续贷”、“农旅贷”等特色产品；债券承销规模大幅增长，债券发行只数跃升北京地区第一位。　同时，该行零售业务继续深化转型。上半年末，客户数达1963万，较去年同期增长218万户，增幅12%，资金量突破6500亿元，同比增长950亿元，增幅17%；储蓄、个贷市场份额保持双提升,北京地区储蓄规模增量排名同业第一。金融市场业务持续推进转型升级，高收益资产占比较年初提升3.4个百分点，核心资产加权收益率较年初提升60个BP，轻资本业务加速发展；联合SWIFT及花旗银行、ING等国际知名金融机构，发布“丝路汇通”跨境金融服务品牌，为中国企业“走出去”提供多元金融支持。未来在新一轮金融对外开放不断深化的背景下，北京银行还将继续抢抓机遇，不断拓宽、深化与外资股东的战略合作，加快推进与ING合资设立法人直销银行等工作。全方位服务小微积极支持实体经济  上半年，北京银行持续加大普惠金融贷款投放，积极支持小微企业主、个体工商户发展，普惠金融贷款余额达到470亿元，增速23%；与远方网等民俗旅游专业运营公司合作拓展农宅宝“互联网+旅游”模式，合力打造山楂小院、黄栌花开等一批精品乡村旅游品牌；持续完善“富民直通车”惠民金融服务体系，与京东、联通等合作拓展新型“富民直通车”服务站点，已在京津冀三地设立300余家农村金融服务站，打通农村惠民金融服务的“最后一米”。　科技金融方面，该行在中关村园区设立30年之际，成立总行级科技金融创新中心，推出“前沿科技贷”产品，科技金融品牌继续引领时代潮流；截至2018年6月末，科技金融贷款余额1410亿元，较年初增加230亿元，增速为19%。文化金融方面，为小微文创企业量身打造“文租贷”合作方案，推出“雍和印象”品牌；截至2018年6月末，文化金融贷款余额694亿元，较年初增加126亿元，增速22%，市场份额始终位居北京市首位。　全力服务国家发展战略区域布局迈出新步伐　北京银行深入推进“建设十大银行”战略目标，积极服务国家三大战略和北京“四个中心”战略定位、城市副中心建设。助力“一带一路”建设，发行国内首张以丝绸之路命名的主题卡“丝路卡”，涵盖借记卡及信用卡两大系列，将区域特色和国家发展战略相结合，提升银行卡品牌区域影响力。银行卡发卡量突破2500万张。区域布局再下一城，宁波分行获批筹建，服务长江经济带的能力进一步增强，成为继青岛分行之后又一个在计划单列市设立的经营机构；截至2018年6月末，已在全国12个主要省市设立14家一级分行和10家二级分行，全行分支机构总数达到618家，并在全国多地拥有10家村镇银行和1家农商银行，金融服务的版图日趋完善。　同时，该行秉承“普之城乡、惠之于民”的社会责任，持续加大惠民金融服务力度，探索出一条颇具特色的普惠金融发展之路。　其践行“绿水青山就是金山银山”的发展理念，推出“农旅贷”特色产品，支持休闲农业和乡村旅游发展；与北京市教委签署全面战略合作协议，出资设立“乡村教师奖励基金”，并继续为“紫禁杯”优秀班主任评选项目提供资金支持；在新疆丝绸之路经济带核心区发行惠民金融产品—“丝绸之路卡”，并为和田“大爱基金”捐赠。多元化产品服务百姓安居乐业，上半年累计投放个人贷款600亿元，满足65万客户安居乐业需求；对接公积金、住建部、链家平台，与万科等品牌开发商合作探索长租贷业务，满足住房刚性需求；通过自主研发产品和联合，协力发展线上小额消费贷，满足客户真实消费的资金需求，满足消费升级需求；发展微贷宝等业务，为个体工商户自主创业提供免抵押、免担保的信用贷款。　筑牢风险防线资产质量保持优秀水平　北京银行深入落实银行业市场乱象治理和监管整改的各项要求，始终把风险防控作为重中之重，推进全机构风险并表管理，强化风险监测预警，跨条线风险联动机制不断完善，初步建立起国内同业第一家具备全流程业务实时监控、全口径数据风险监测、全机构信息互动等功能的风控指挥监测中心。截至2018年6月末，北京银行不良贷款率为1.23%，较年初下降0.01个百分点，拨备覆盖率260.48%，拨贷比3.20%。资产质量和风险抵御能力始终保持上市银行优秀水平。坚持创新引领金融科技核心竞争力增强　北京银行聚焦智能数据、强大算力、科学算法三大能力，夯实渠道做活、中台做大、后台做强三大支柱，以人工智能、分布式数据库、大数据、等作为关键性技术驱动，上半年完成293个重大项目、创新产品敏捷迭代开发投产。夯实金融科技基础设施，高标准建设顺义科技研发中心，打造引领创新发展的“战略高地”；研究并构建分布式数据库，打造北京银行金融服务云，建设充分适应互联网环境的信息系统应用架构基础。该行还牵手京东、小米、腾讯等互联网公司，联合浙江大学计算机辅助设计与图形学国家重点实验室等高校与科研机构联手打造科技金融新生态。积极筹建北京银行金融科技公司，将通过有效的产品创新激励机制和强大的产品研发能力，提供具有竞争力的金融科技产品和解决方案，实现金融科技能力的广泛实践，全力打造业界领先的银行系科技公司。　新时代是奋斗者的时代，站在新时代的历史起点上，北京银行董事长张东宁表示，北京银行将始终以习近平新时代中国特色社会主义思想为指导，紧紧围绕高质量发展这一根本要求，继续致力于建设“十大银行”，以坚实的金融科技为转型发展赋能，持续加快经营转型步伐，深入推进集约化、专业化、精细化管理，为打造永续发展、行稳致远的百年老店不懈奋斗。'

# 使用jieba的分词和NER模块提取人名和职务
jieba.enable_paddle()
words = pseg.cut(text,use_paddle=True)

names = []
names_index = []
word_all = []
titles = []
titles_index = []
flag_all = []
job_index = []
job_text = []


# for i in range(len(words)):
#     print(words[i])
for word, flag in words:
    word_all.append(word)
    flag_all.append(flag)

for i in range(len(word_all)):
    if flag_all[i] in title_tag:
        titles.append(word_all[i])
        titles_index.append(i)
    elif flag_all[i] in person_tag:
        if word_all[i] not in names:
            names.append(word_all[i])
            names_index.append(i)


for i in range(len(names_index)):
    for j in range(len(titles_index)):
        if titles_index[j] < names_index[i]:
            if set(flag_all[titles_index[j]+1:names_index[i]]).issubset(set(job_tag)):
                if len(job_text)!=0:
                    for element in job_text:
                        if "".join(word_all[titles_index[j]:names_index[i]+1]) not in element:
                            job_text.append("".join(word_all[titles_index[j]:names_index[i]+1]))
                else:
                    job_text.append("".join(word_all[titles_index[j]:names_index[i]+1]))
        
names = [name for name in names if len(name) >= 2 and re.search("[\u4e00-\u9fa5]", name)]
titles = set([title for title in titles if len(title) >= 4 and re.search("[\u4e00-\u9fa5]", title)])
tr4s = TextRank4Sentence()
tr4s.analyze(text=text, lower=True, source='all_filters')


# # 打印结果
print(job_text)
print(names)
print(titles)
# 输出摘要
print('摘要：')
for item in tr4s.get_key_sentences(num=3):
    print(item.sentence)

In [ ]:
from textrank4zh import TextRank4Sentence

text = "中国人民银行支付结算司司长谢众14日在中信“党费通”产品发布暨客户签约仪式上指出，2018年中国人民银行将继续大力推动我国非现金支付环境的建设，指导市场主体对企业、居民提供更加安全、便捷、高效的支付产品。同时，持续做好正本清源工作，进一步加大市场监管力度，防范化解金融风险，对各类违规支付行为的整顿处罚绝不手软，维护支付市场的公平竞争秩序，使移动支付服务更好地发挥对实体经济和普惠金融发展的促进作用。谢众表示，中信“党费通”率先使用移动支付技术实现党费交纳和管理，为各级党组织和广大党员提供创新服务，实现了为新时期党建工作赋能。中国人民银行一直致力于推动我国移动支付服务的创新发展，指导主导市场主体持续推动非现金支付工具的应用。去年以来，中国银联和商业银行正在积极组织开展移动支付便民示范工程，统筹推进移动支付便民利用和受理环境改造等工作，部署打造银行业统一移动支付APP，广泛覆盖各类移动终端产品和移动支付技术，深度结合地铁、公交、校企园区和公用事业缴费等人民群众衣食住行各类支付产品，推动普惠金融发展。中信银行党委书记、董事长李庆萍介绍，作为集党费缴纳、管理、教育、服务于一体的综合管理平台，“党费通”在实现多场景运用和多方式支付的同时，兼具安全性和仪式感，是中信集团和中信银行打造智慧党建的重要成果。在14日的签约环节，中信银行与10家“党费通”产品使用单位和党建工作平台建设单位分别签约。"

tr4s = TextRank4Sentence()
tr4s.analyze(text=text, lower=True, source='all_filters')

# 输出摘要 - 前3句话
print('摘要：')
for item in tr4s.get_key_sentences(num=3): 
    print(item.sentence)
